# Dhaka GPS Trace -> Travel Time Prediction: Full Training Pipeline

This notebook reproduces, end to end, the pipeline used to build the Dhaka travel-time-prediction training dataset and model:

1. Parse merged real GPS traces, drop junk traces, split into individual trips by time gap
2. Speed-based cleaning + minimum-trip validity filter
3. Sub-trajectory sampling (multi-scale) -> raw (src, dst, duration) rows
4. OSRM routing engine setup (Docker) -- reference commands, run once outside the notebook
5. Geometric + road-network feature computation (via OSRM)
6. Spatial zone model (fixed geographic grid -- see Step 6 for why this replaced KMeans)
7. Time features (cyclical hour, day-of-week, holiday, rush-hour, traffic-period bucket)
8. Grouped train/val/test split by trip_id
9. Historical traffic-factor feature (reference version, train-only, with fallback)
10. Assemble final dataset (+ combined full_dataset.csv)
11. Finalize model feature list
12. Baseline models
13. Train XGBoost with grouped 5-fold CV (historical feature recomputed per fold)
14. Evaluate vs. baselines
15. Feature importance
16. Algorithm comparison (Ridge / Random Forest / XGBoost)
17. Persist inference artifacts (models + grid params + historical tables + schema)
18. Inference-time feature pipeline, demonstrated live
19. Try your own src/dst/time

**Data paths** (adjust `PROJECT_DIR` below if you move the project folder):
- Input: `gps-trace data/dhaka_traces_merged.gpx`
- Output (datasets): `gps-trace data/training_pipeline/*.csv`
- Output (trained models): `trained-model/xgb_model.joblib`, `trained-model/inference_bundle.joblib`

**Prerequisite**: Step 4's OSRM server must be running before executing the Step 5 cell (see that section for the Docker commands used).

In [5]:
import os, re, math, json, csv, bisect, statistics, random, time
from datetime import datetime

import numpy as np
import pandas as pd
from xml.etree import ElementTree as ET

PROJECT_DIR = os.path.abspath("..")  # repo root -- this notebook lives in <repo>/notebooks/
GPS_DIR = os.path.join(PROJECT_DIR, "gps-trace data")
OUT_DIR = os.path.join(GPS_DIR, "training_pipeline")
MODEL_DIR = os.path.join(PROJECT_DIR, "trained-model")   # all trained model artifacts live here
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

MERGED_GPX = os.path.join(GPS_DIR, "dhaka_traces_merged.gpx")
NS = "{http://www.topografix.com/GPX/1/0}"

print("PROJECT_DIR:", PROJECT_DIR)
print("Merged GPX exists:", os.path.exists(MERGED_GPX))
print("MODEL_DIR:", MODEL_DIR)

PROJECT_DIR: d:\Academic\4-1\Capstone\GPS_TRACE_ML
Merged GPX exists: True
MODEL_DIR: d:\Academic\4-1\Capstone\GPS_TRACE_ML\trained-model


## Step 1 -- Parse merged GPX, drop junk traces, split into trips by time gap

Drops the 4 confirmed junk traces (flight-sim / spam uploads) and any pre-2000 corrupted
timestamps, then splits each trace into separate trip segments wherever the gap between
consecutive points exceeds `TIME_GAP_S` (10 minutes).

In [6]:
JUNK_TRACE_IDS = {"12226010", "12227373", "12374266", "12374551"}
TIME_GAP_S = 600  # 10 minutes

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlmb / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

def parse_time(t):
    return datetime.fromisoformat(t.replace("Z", "+00:00"))

# Stream the 40+ MB file one <trk> at a time (a full ET.parse tree does not fit in RAM here).
traces = []
for _, trk in ET.iterparse(MERGED_GPX, events=("end",)):
    if trk.tag != f"{NS}trk":
        continue
    url_el = trk.find(f"{NS}url")
    name_el = trk.find(f"{NS}name")
    trace_id = None
    if url_el is not None and url_el.text:
        m = re.search(r"traces/(\d+)", url_el.text)
        if m:
            trace_id = m.group(1)
    name = name_el.text if name_el is not None else ""
    pts = []
    for trkseg in trk.findall(f"{NS}trkseg"):
        for pt in trkseg.findall(f"{NS}trkpt"):
            lat = float(pt.get("lat")); lon = float(pt.get("lon"))
            time_el = pt.find(f"{NS}time")
            if time_el is None:
                continue
            try:
                dt = parse_time(time_el.text)
            except Exception:
                continue
            pts.append((lat, lon, dt))
    pts.sort(key=lambda p: p[2])
    traces.append({"trace_id": trace_id, "name": name, "points": pts})
    trk.clear()

print(f"Loaded {len(traces)} traces from merged file")

split_trips = []
dropped_junk = 0
for tr in traces:
    if tr["trace_id"] in JUNK_TRACE_IDS:
        dropped_junk += 1
        continue
    pts = [p for p in tr["points"] if p[2].year >= 2000]
    if len(pts) < 2:
        continue
    current = [pts[0]]
    for prev, curr in zip(pts, pts[1:]):
        gap = (curr[2] - prev[2]).total_seconds()
        if gap > TIME_GAP_S or gap < 0:
            if len(current) >= 2:
                split_trips.append({"trace_id": tr["trace_id"], "name": tr["name"], "points": current})
            current = [curr]
        else:
            current.append(curr)
    if len(current) >= 2:
        split_trips.append({"trace_id": tr["trace_id"], "name": tr["name"], "points": current})

print(f"Dropped {dropped_junk} junk traces")
print(f"After time-gap splitting (gap > {TIME_GAP_S}s): {len(split_trips)} trip segments")

durations = [(t["points"][-1][2] - t["points"][0][2]).total_seconds() for t in split_trips]
point_counts = [len(t["points"]) for t in split_trips]
print(f"duration(s): min={min(durations):.0f} median={statistics.median(durations):.0f} max={max(durations):.0f}")
print(f"points/trip: min={min(point_counts)} median={statistics.median(point_counts):.0f} max={max(point_counts)}")

Loaded 89 traces from merged file
Dropped 4 junk traces
After time-gap splitting (gap > 600s): 295 trip segments
duration(s): min=1 median=2390 max=26671
points/trip: min=2 median=848 max=15027


## Step 2 -- Glitch-based splitting + minimum-trip validity filter

Splits further wherever consecutive-point speed exceeds 120 km/h (GPS glitch), then drops
trips that are too short to be meaningful (< 5 points, < 60s duration, or < 300m total
distance). Saves the surviving cleaned trips to `cleaned_trips.json` for reuse by later steps.

In [7]:
SPEED_GLITCH_MS = 120 * 1000 / 3600.0  # 120 km/h -> m/s
MIN_POINTS = 5
MIN_DIST_M = 300
MIN_DUR_S = 60
# A trip whose 90th-percentile point-to-point speed stays below this never moved like a motor
# vehicle (walkers peak ~5-9 km/h; a car stuck in Dhaka traffic still reaches 15+ between stops).
MIN_P90_SPEED_KMH = 12.0

def trip_total_distance(pts):
    d = 0.0
    for a, b in zip(pts, pts[1:]):
        d += haversine_m(a[0], a[1], b[0], b[1])
    return d

glitch_split_trips = []
glitch_splits = 0
for t in split_trips:
    pts = t["points"]
    current = [pts[0]]
    for prev, curr in zip(pts, pts[1:]):
        dt = (curr[2] - prev[2]).total_seconds()
        if dt <= 0:
            current.append(curr)
            continue
        dist = haversine_m(prev[0], prev[1], curr[0], curr[1])
        if dist / dt > SPEED_GLITCH_MS:
            glitch_splits += 1
            if len(current) >= 2:
                glitch_split_trips.append({**t, "points": current})
            current = [curr]
        else:
            current.append(curr)
    if len(current) >= 2:
        glitch_split_trips.append({**t, "points": current})

print(f"Additional glitch-based splits: {glitch_splits}")
print(f"Trip segments after glitch-splitting: {len(glitch_split_trips)}")

def p90_point_speed_kmh(pts):
    speeds = []
    for a, b in zip(pts, pts[1:]):
        dt = (b[2] - a[2]).total_seconds()
        if dt > 0:
            speeds.append(haversine_m(a[0], a[1], b[0], b[1]) / dt * 3.6)
    return float(np.percentile(speeds, 90)) if speeds else 0.0

valid_trips = []
dropped_non_motorized = 0
for t in glitch_split_trips:
    pts = t["points"]
    if len(pts) < MIN_POINTS:
        continue
    dur = (pts[-1][2] - pts[0][2]).total_seconds()
    if dur < MIN_DUR_S:
        continue
    dist = trip_total_distance(pts)
    if dist < MIN_DIST_M:
        continue
    if p90_point_speed_kmh(pts) < MIN_P90_SPEED_KMH:
        dropped_non_motorized += 1
        continue
    avg_speed_kmh = (dist / 1000.0) / (dur / 3600.0)
    valid_trips.append({**t, "total_distance_m": dist, "total_duration_s": dur, "avg_speed_kmh": avg_speed_kmh})

for idx, v in enumerate(valid_trips):
    v["trip_id"] = f"{v['trace_id'] or 'noid'}_{idx}"

print(f"Dropped {dropped_non_motorized} non-motorized trips (p90 point speed < {MIN_P90_SPEED_KMH:.0f} km/h)")
print(f"Valid trips remaining: {len(valid_trips)}")
speeds = [v["avg_speed_kmh"] for v in valid_trips]
print(f"avg speed (km/h): min={min(speeds):.1f} median={statistics.median(speeds):.1f} max={max(speeds):.1f}")

serializable_trips = [{
    "trip_id": v["trip_id"], "trace_id": v["trace_id"], "name": v["name"],
    "total_distance_m": v["total_distance_m"], "total_duration_s": v["total_duration_s"],
    "avg_speed_kmh": v["avg_speed_kmh"],
    "points": [[lat, lon, dt.isoformat()] for lat, lon, dt in v["points"]],
} for v in valid_trips]
with open(os.path.join(OUT_DIR, "cleaned_trips.json"), "w", encoding="utf-8") as f:
    json.dump(serializable_trips, f)
print(f"Saved {len(serializable_trips)} cleaned trips -> cleaned_trips.json")

Additional glitch-based splits: 144
Trip segments after glitch-splitting: 347
Dropped 53 non-motorized trips (p90 point speed < 12 km/h)
Valid trips remaining: 258
avg speed (km/h): min=2.3 median=15.7 max=68.2
Saved 258 cleaned trips -> cleaned_trips.json


## Step 3 -- Multi-scale sub-trajectory sampling

For each cleaned trip, samples anchor points and, from each anchor, generates pairs reaching
several target distance scales (0.3, 1, 3, 8 km, plus the trip's own endpoint) -- this is the
fix applied after the first version produced only ~200m micro-hops. An implied-speed QC filter
(1-100 km/h using straight-line distance) drops remaining bad pairs.

In [8]:
TARGET_KMS = [0.3, 1.0, 3.0, 8.0]
MIN_SEP_S = 60
MAX_ANCHORS = 12
MIN_IMPLIED_SPEED_KMH = 1.0
MAX_IMPLIED_SPEED_KMH = 100.0

rows = []
row_id = 0
dropped_qc_speed = 0
random.seed(42)

for t in valid_trips:
    pts = t["points"]
    n = len(pts)
    trip_id = t["trip_id"]
    if n < 2:
        continue

    cumdist = [0.0] * n
    for i in range(1, n):
        cumdist[i] = cumdist[i - 1] + haversine_m(pts[i - 1][0], pts[i - 1][1], pts[i][0], pts[i][1])

    raw_anchor_positions = sorted(set(
        int(round(i * (n - 2) / max(1, MAX_ANCHORS - 1))) for i in range(MAX_ANCHORS)
    )) if n > 2 else [0]
    anchors = [a for a in raw_anchor_positions if a <= n - 2]

    for a in anchors:
        candidate_js = set()
        for target_km in TARGET_KMS:
            target_cum = cumdist[a] + target_km * 1000.0
            j = bisect.bisect_left(cumdist, target_cum, a + 1, n)
            if j >= n:
                j = n - 1
            candidate_js.add((j, target_km))
        candidate_js.add((n - 1, "endpoint"))

        seen_j = set()
        for j, source_target in sorted(candidate_js, key=lambda x: x[0]):
            if j <= a or j in seen_j:
                continue
            seen_j.add(j)
            dt = (pts[j][2] - pts[a][2]).total_seconds()
            if dt < MIN_SEP_S:
                continue
            dist_direct_km = haversine_m(pts[a][0], pts[a][1], pts[j][0], pts[j][1]) / 1000.0
            implied_speed_kmh = dist_direct_km / (dt / 3600.0)
            if implied_speed_kmh < MIN_IMPLIED_SPEED_KMH or implied_speed_kmh > MAX_IMPLIED_SPEED_KMH:
                dropped_qc_speed += 1
                continue
            rows.append({
                "row_id": row_id, "trip_id": trip_id,
                "src_lat": pts[a][0], "src_lon": pts[a][1], "src_time": pts[a][2].isoformat(),
                "dst_lat": pts[j][0], "dst_lon": pts[j][1], "dst_time": pts[j][2].isoformat(),
                "duration_seconds": dt,
            })
            row_id += 1

print(f"Total raw (src,dst,duration) rows generated: {len(rows)}")
print(f"Rows dropped by implied-speed QC filter: {dropped_qc_speed}")

step3_df = pd.DataFrame(rows)
step3_df.to_csv(os.path.join(OUT_DIR, "step3_raw_pairs.csv"), index=False)
print(f"Saved {len(step3_df)} rows -> step3_raw_pairs.csv")

Total raw (src,dst,duration) rows generated: 8258
Rows dropped by implied-speed QC filter: 525
Saved 8258 rows -> step3_raw_pairs.csv


## Step 4 -- OSRM routing engine (Docker) -- run once, outside this notebook

The project already had a Bangladesh OSM extract in `osrm-data/` (raw `.osm.pbf` +
pre-processed `.osrm.*` files). The pre-processed files didn't match the current OSRM
Docker image's version ("fingerprint mismatch"), so the graph was rebuilt from the raw
`.osm.pbf` and served on **host port 5050** (5000 was already taken by another container
on this machine). Run these once in a terminal before executing the Step 5 cell below:

```bash
docker pull osrm/osrm-backend:latest

# Rebuild the routing graph from the raw extract (only needed once, or if osrm-data changes)
docker run --rm -v "/path/to/osrm-data:/data" osrm/osrm-backend osrm-extract -p /opt/car.lua /data/bangladesh-latest.osm.pbf
docker run --rm -v "/path/to/osrm-data:/data" osrm/osrm-backend osrm-partition /data/bangladesh-latest.osrm
docker run --rm -v "/path/to/osrm-data:/data" osrm/osrm-backend osrm-customize /data/bangladesh-latest.osrm

# Serve it
docker run -d --name dhaka-osrm -p 5050:5000 -v "/path/to/osrm-data:/data" osrm/osrm-backend osrm-routed --algorithm mld /data/bangladesh-latest.osrm
```

The cell below just checks the server is reachable before Step 5 proceeds.

In [9]:
import requests

OSRM_BASE = "http://localhost:5050"
try:
    r = requests.get(f"{OSRM_BASE}/route/v1/driving/90.3757,23.7389;90.3622,23.7565?overview=false", timeout=5)
    ok = r.json().get("code") == "Ok"
except Exception as e:
    ok = False
    print("OSRM not reachable:", e)

print("OSRM reachable and responding:" if ok else "OSRM NOT reachable -- start the Docker container from the cell above first.", ok)

OSRM reachable and responding: True


## Step 5 -- Geometric + road-network features

Computes `haversine_distance_km` / `bearing_degrees` locally, then queries OSRM (threaded)
for `osrm_route_distance_km` / `osrm_free_flow_duration_sec` for every row, and derives
`route_directness_ratio`. Two cleaning filters are then applied based on what OSRM revealed:
drop rows where the OSRM-implied road speed exceeds 100 km/h, and rows where OSRM's route
distance came back shorter than the straight-line distance (a snapping glitch) -- both catch
GPS-glitch pairs that the Step 3 straight-line-only filter couldn't.

In [10]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def haversine_km(lat1, lon1, lat2, lon2):
    return haversine_m(lat1, lon1, lat2, lon2) / 1000.0

def bearing_deg(lat1, lon1, lat2, lon2):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dlmb = math.radians(lon2 - lon1)
    x = math.sin(dlmb) * math.cos(p2)
    y = math.cos(p1) * math.sin(p2) - math.sin(p1) * math.cos(p2) * math.cos(dlmb)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

df5 = pd.read_csv(os.path.join(OUT_DIR, "step3_raw_pairs.csv"))
df5["haversine_distance_km"] = df5.apply(lambda r: haversine_km(r.src_lat, r.src_lon, r.dst_lat, r.dst_lon), axis=1)
df5["bearing_degrees"] = df5.apply(lambda r: bearing_deg(r.src_lat, r.src_lon, r.dst_lat, r.dst_lon), axis=1)

session = requests.Session()

def query_osrm(idx, src_lat, src_lon, dst_lat, dst_lon):
    url = f"{OSRM_BASE}/route/v1/driving/{src_lon},{src_lat};{dst_lon},{dst_lat}?overview=false"
    try:
        data = session.get(url, timeout=5).json()
        if data.get("code") == "Ok":
            route = data["routes"][0]
            return idx, route["distance"] / 1000.0, route["duration"], None
        return idx, None, None, data.get("code", "unknown_error")
    except Exception as e:
        return idx, None, None, str(e)

n = len(df5)
osrm_dist, osrm_dur, osrm_err = [None] * n, [None] * n, [None] * n
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    futures = [ex.submit(query_osrm, i, row.src_lat, row.src_lon, row.dst_lat, row.dst_lon)
               for i, row in df5.iterrows()]
    for fut in as_completed(futures):
        idx, dist, dur, err = fut.result()
        osrm_dist[idx], osrm_dur[idx], osrm_err[idx] = dist, dur, err
print(f"OSRM queries: {n} rows in {time.time()-t0:.1f}s")

df5["osrm_route_distance_km"] = osrm_dist
df5["osrm_free_flow_duration_sec"] = osrm_dur
success = df5["osrm_route_distance_km"].notna()
print(f"OSRM success rate: {success.sum()}/{n}")

df5["route_directness_ratio"] = np.nan
valid = success & (df5["osrm_route_distance_km"] > 0)
df5.loc[valid, "route_directness_ratio"] = df5.loc[valid, "haversine_distance_km"] / df5.loc[valid, "osrm_route_distance_km"]

df5 = df5.loc[valid].copy()
df5["implied_road_speed_kmh"] = df5["osrm_route_distance_km"] / (df5["duration_seconds"] / 3600.0)

bad_speed = df5["implied_road_speed_kmh"] > 100
bad_snap = df5["osrm_route_distance_km"] < df5["haversine_distance_km"] * 0.98
drop_mask = bad_speed | bad_snap
print(f"Dropping {bad_speed.sum()} rows (implied road speed > 100 km/h) and "
      f"{bad_snap.sum()} rows (OSRM shorter than haversine, snap glitch)")

df5_clean = df5.loc[~drop_mask].drop(columns=["implied_road_speed_kmh"]).copy()
df5_clean.to_csv(os.path.join(OUT_DIR, "step5_features_clean.csv"), index=False)
print(f"Saved {len(df5_clean)} clean rows -> step5_features_clean.csv")

OSRM queries: 8258 rows in 10.5s
OSRM success rate: 8258/8258
Dropping 298 rows (implied road speed > 100 km/h) and 31 rows (OSRM shorter than haversine, snap glitch)
Saved 7926 clean rows -> step5_features_clean.csv


## Step 6 -- Spatial zone model (fixed geographic grid)

**Updated after a live A/B test.** Originally zones were fit with KMeans (data-driven
clusters on wherever the training points happened to be). After building a proper comparison
-- same grouped 5-fold CV, same model, same features, only the zoning method differs -- a
**fixed geographic grid** covering the whole bounding box of the data won on every metric:

| Metric | KMeans (old) | Fixed Grid (new) |
|---|---|---|
| MAE | 658.8s | **527.1s** |
| RMSE | 1988.4s | **1216.9s** |
| MAPE | 67.2% | **62.3%** |

Adopted as the new default. The grid divides the full lat/lon bounding box of all src/dst
points into `N_ROWS x N_COLS` = 20 fixed cells. Unlike KMeans, a point's zone is determined
purely by which literal geographic cell its coordinates fall into -- independent of where
training data happened to be, so the zone's *meaning* stays geographically stable rather than
shifting with data density. The real cost: this produced 2 zones with zero real points behind
them (KMeans never has empty zones, by construction) -- but it still won decisively.

In [11]:
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv(os.path.join(OUT_DIR, "step5_features_clean.csv"))
print(f"Loaded {len(df)} clean rows, {df.trip_id.nunique()} distinct trips")

# Fixed geographic grid: divide the bounding box of ALL src+dst points into N_ROWS x N_COLS
# cells. Zone assignment is a pure function of (lat, lon) -- independent of KMeans / where
# training data happened to be. Reused later (Step 17) for the deployed inference bundle.
N_ROWS, N_COLS = 4, 5  # 20 cells total, matching the original KMeans zone count
all_lats = pd.concat([df["src_lat"], df["dst_lat"]])
all_lons = pd.concat([df["src_lon"], df["dst_lon"]])
GRID_MIN_LAT, GRID_MAX_LAT = all_lats.min(), all_lats.max()
GRID_MIN_LON, GRID_MAX_LON = all_lons.min(), all_lons.max()
GRID_LAT_STEP = (GRID_MAX_LAT - GRID_MIN_LAT) / N_ROWS
GRID_LON_STEP = (GRID_MAX_LON - GRID_MIN_LON) / N_COLS
print(f"Grid bounding box: lat [{GRID_MIN_LAT:.4f}, {GRID_MAX_LAT:.4f}]  "
      f"lon [{GRID_MIN_LON:.4f}, {GRID_MAX_LON:.4f}]  ({N_ROWS}x{N_COLS} = {N_ROWS*N_COLS} cells)")

def grid_cell(lat, lon):
    r = min(int((lat - GRID_MIN_LAT) / GRID_LAT_STEP), N_ROWS - 1)
    c = min(int((lon - GRID_MIN_LON) / GRID_LON_STEP), N_COLS - 1)
    return r * N_COLS + c

df["src_zone_id"] = df.apply(lambda row: grid_cell(row.src_lat, row.src_lon), axis=1)
df["dst_zone_id"] = df.apply(lambda row: grid_cell(row.dst_lat, row.dst_lon), axis=1)
df["od_zone_pair_id"] = df["src_zone_id"].astype(str) + "_" + df["dst_zone_id"].astype(str)

zone_counts = pd.concat([df["src_zone_id"], df["dst_zone_id"]]).value_counts()
empty_zones = sorted(set(range(N_ROWS * N_COLS)) - set(zone_counts.index))
print(f"Zone sizes: min={zone_counts.min()} median={zone_counts.median():.0f} max={zone_counts.max()}  "
      f"zones_with_data={zone_counts.index.nunique()}/{N_ROWS*N_COLS}  empty_zones={empty_zones}")
print(f"{df['od_zone_pair_id'].nunique()} distinct OD zone-pairs")

Loaded 7926 clean rows, 255 distinct trips
Grid bounding box: lat [23.5501, 24.2000]  lon [90.2001, 90.6178]  (4x5 = 20 cells)
Zone sizes: min=1 median=232 max=10504  zones_with_data=18/20  empty_zones=[15, 19]
79 distinct OD zone-pairs


## Step 7 -- Time features

Cyclical hour encoding, day-of-week (Bangladesh weekend = Friday+Saturday), a fixed-date
national holiday flag (lunar Islamic holidays like Eid are **not** covered -- no offline
calendar source was available), a rush-hour flag, and the `traffic_period_bucket` categorical
(the friend's suggested time-period buckets, turned into a feature).

In [12]:
FIXED_HOLIDAYS_MD = {(2, 21), (3, 26), (4, 14), (5, 1), (8, 15), (12, 16), (12, 25)}
# NOTE: lunar Islamic holidays (Eid ul-Fitr, Eid ul-Adha, Ashura, etc.) are NOT included --
# they shift every year and no offline calendar source was available. Known limitation.

# GPX timestamps are UTC; all time features must be Dhaka local time (UTC+6) -- inference
# queries are given in local time.
df["src_time_parsed"] = pd.to_datetime(df["src_time"], utc=True).dt.tz_convert("Asia/Dhaka")

def hour_frac(dt):
    return dt.hour + dt.minute / 60.0 + dt.second / 3600.0

def traffic_bucket(h):
    if h < 6: return "night"
    elif h < 8: return "morning_offpeak"
    elif h < 10: return "morning_rush"
    elif h < 17: return "midday"
    elif h < 21: return "evening_rush"
    else: return "evening_winddown"

hours = df["src_time_parsed"].apply(hour_frac)
df["hour_sin"] = np.sin(2 * np.pi * hours / 24.0)
df["hour_cos"] = np.cos(2 * np.pi * hours / 24.0)
df["day_of_week"] = df["src_time_parsed"].dt.day_name()
df["is_friday"] = df["day_of_week"] == "Friday"
df["is_saturday"] = df["day_of_week"] == "Saturday"
df["is_weekend"] = df["is_friday"] | df["is_saturday"]
df["rush_hour_flag"] = hours.apply(lambda h: (8 <= h < 10) or (17 <= h < 21))
df["traffic_period_bucket"] = hours.apply(traffic_bucket)
df["is_holiday"] = df["src_time_parsed"].apply(lambda dt: (dt.month, dt.day) in FIXED_HOLIDAYS_MD)

print("Time features computed. traffic_period_bucket counts:")
print(df["traffic_period_bucket"].value_counts())

Time features computed. traffic_period_bucket counts:
traffic_period_bucket
midday              3523
evening_rush        2691
evening_winddown     869
morning_rush         367
night                363
morning_offpeak      113
Name: count, dtype: int64


## Step 8 -- Grouped train/val/test split (by `trip_id`)

Ensures no single real trip contributes rows to more than one split, so evaluation isn't
inflated by near-duplicate rows from the same trip leaking across splits.

In [13]:
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
trainval_idx, test_idx = next(gss1.split(df, groups=df["trip_id"]))
df_trainval = df.iloc[trainval_idx]
df_test = df.iloc[test_idx].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
train_idx, val_idx = next(gss2.split(df_trainval, groups=df_trainval["trip_id"]))
df_train = df_trainval.iloc[train_idx].copy()
df_val = df_trainval.iloc[val_idx].copy()

print(f"Split sizes: train={len(df_train)} val={len(df_val)} test={len(df_test)}")
assert not (set(df_train.trip_id) & set(df_val.trip_id))
assert not (set(df_train.trip_id) & set(df_test.trip_id))
assert not (set(df_val.trip_id) & set(df_test.trip_id))
print("Verified: zero trip_id overlap between splits")

Split sizes: train=5460 val=1274 test=1192
Verified: zero trip_id overlap between splits


## Step 9 -- Historical traffic-factor feature (reference version, train-only, with fallback)

`historical_avg_speed_kmh` is the measured average road speed for each
`(od_zone_pair_id, traffic_period_bucket)` combination, computed **only from the training
split** and joined onto all three splits (to avoid leakage). Falls back to a coarser estimate
when a specific combination has too few training samples.

**Update**: this static, Step-8-split-based version is kept here for reference and is what
gets saved into `final_*.csv` / `full_dataset.csv` for convenience. The *official* CV
evaluation in Steps 11-16 below no longer relies on it -- it recomputes this feature fresh
inside each CV fold instead (leakage-safe against the *actual* fold boundaries used for
training, not this older split), which is the methodology that was validated in the KMeans
vs. fixed-grid comparison in Step 6.

**Known caveat** (found during review): thin buckets like `evening_rush` (only ~19 real trips)
can be dominated by a single unusual trip (e.g. a highway/bypass trip), producing a value that
doesn't match real Dhaka traffic behaviour. Documented here rather than silently trusted.

In [14]:
df_train = df_train.copy()
df_train["true_speed_kmh"] = df_train["osrm_route_distance_km"] / (df_train["duration_seconds"] / 3600.0)

# Support is counted in DISTINCT TRIPS (one trip yields many multi-scale rows), speed is the median.
MIN_SUPPORT = 3
agg = dict(speed=("true_speed_kmh", "median"), n_trips=("trip_id", "nunique"))
lvl1 = df_train.groupby(["od_zone_pair_id", "traffic_period_bucket"]).agg(**agg)
lvl2 = df_train.groupby(["od_zone_pair_id"]).agg(**agg)
lvl3 = df_train.groupby(["traffic_period_bucket"]).agg(**agg)
global_speed = df_train["true_speed_kmh"].median()

def period_factor(bucket, lvl3, global_speed):
    """How much faster/slower this time of day is than the all-day median (1.0 if unsupported)."""
    if bucket in lvl3.index and lvl3.loc[bucket, "n_trips"] >= MIN_SUPPORT:
        return lvl3.loc[bucket, "speed"] / global_speed
    return 1.0

def lookup_historical_speed(row):
    key1 = (row["od_zone_pair_id"], row["traffic_period_bucket"])
    if key1 in lvl1.index and lvl1.loc[key1, "n_trips"] >= MIN_SUPPORT:
        return lvl1.loc[key1, "speed"], "od_pair+period"
    key2 = row["od_zone_pair_id"]
    if key2 in lvl2.index and lvl2.loc[key2, "n_trips"] >= MIN_SUPPORT:
        return lvl2.loc[key2, "speed"] * period_factor(row["traffic_period_bucket"], lvl3, global_speed), "od_pair_only"
    key3 = row["traffic_period_bucket"]
    if key3 in lvl3.index and lvl3.loc[key3, "n_trips"] >= MIN_SUPPORT:
        return lvl3.loc[key3, "speed"], "period_only"
    return global_speed, "global_fallback"

for name, d in [("train", df_train), ("val", df_val), ("test", df_test)]:
    results = d.apply(lookup_historical_speed, axis=1)
    d["historical_avg_speed_kmh"] = results.apply(lambda x: x[0])
    d["historical_feature_level"] = results.apply(lambda x: x[1])

print("historical_avg_speed_kmh computed for all splits (train-only aggregation, no nulls):")
for name, d in [("train", df_train), ("val", df_val), ("test", df_test)]:
    print(f"  {name}: nulls={d.historical_avg_speed_kmh.isna().sum()}  level counts={dict(d.historical_feature_level.value_counts())}")

historical_avg_speed_kmh computed for all splits (train-only aggregation, no nulls):
  train: nulls=0  level counts={'od_pair+period': np.int64(4761), 'od_pair_only': np.int64(443), 'period_only': np.int64(256)}
  val: nulls=0  level counts={'od_pair+period': np.int64(954), 'od_pair_only': np.int64(213), 'period_only': np.int64(107)}
  test: nulls=0  level counts={'od_pair+period': np.int64(974), 'od_pair_only': np.int64(122), 'period_only': np.int64(96)}


## Step 10 -- Assemble final dataset

Builds the final feature schema (+ `log_duration` target) for each split, saves
`final_train.csv` / `final_val.csv` / `final_test.csv` (the trip-grouped, leakage-safe
version), and then -- per an explicit later decision -- also concatenates all three into a
single **`full_dataset.csv`**, which is the file actually used for model training below.

Two important design decisions were made explicitly with the project owner along the way:
- `full_dataset.csv` combines all rows with **no persisted train/val/test split** in the file itself.
- Model training below instead uses **grouped 5-fold cross-validation** (by `trip_id`) directly
  on `full_dataset.csv`, which still avoids the same-trip leakage without needing separate split files.

In [15]:
FEATURE_COLS = [
    "trip_id", "row_id",
    "src_lat", "src_lon", "dst_lat", "dst_lon", "src_time",
    "haversine_distance_km", "bearing_degrees",
    "osrm_route_distance_km", "osrm_free_flow_duration_sec", "route_directness_ratio",
    "src_zone_id", "dst_zone_id", "od_zone_pair_id",
    "hour_sin", "hour_cos", "day_of_week", "is_friday", "is_saturday", "is_weekend",
    "is_holiday", "rush_hour_flag", "traffic_period_bucket",
    "historical_avg_speed_kmh", "historical_feature_level",
    "duration_seconds",
]

for name, d in [("train", df_train), ("val", df_val), ("test", df_test)]:
    d["log_duration"] = np.log(d["duration_seconds"])
    out = d[FEATURE_COLS + ["log_duration"]].copy()
    out.to_csv(os.path.join(OUT_DIR, f"final_{name}.csv"), index=False)
    print(f"Saved final_{name}.csv -> rows={len(out)}")

# Combine into one full dataset (explicit later decision -- see markdown above)
full = pd.concat([
    pd.read_csv(os.path.join(OUT_DIR, "final_train.csv")),
    pd.read_csv(os.path.join(OUT_DIR, "final_val.csv")),
    pd.read_csv(os.path.join(OUT_DIR, "final_test.csv")),
], ignore_index=True)
full.to_csv(os.path.join(OUT_DIR, "full_dataset.csv"), index=False)
print(f"\nSaved full_dataset.csv -> rows={len(full)}, distinct trips={full.trip_id.nunique()}")

Saved final_train.csv -> rows=5460
Saved final_val.csv -> rows=1274
Saved final_test.csv -> rows=1192

Saved full_dataset.csv -> rows=7926, distinct trips=255


## Steps 11-15 -- Model input features, baseline, training, evaluation, feature importance

- **Step 11**: finalize the model input feature list (drops bookkeeping columns and raw lat/lon).
- **Step 12**: two baselines -- naive (OSRM free-flow duration as-is) and calibrated (x global
  median slowdown factor).
- **Step 13**: XGBoost, trained with **grouped 5-fold cross-validation by `trip_id`** directly
  on `full_dataset.csv`. `historical_avg_speed_kmh` is recomputed fresh inside each fold (from
  that fold's training rows only) rather than reusing the static Step 9 column -- this is the
  methodology validated in the Step 6 zoning comparison, and fixes a subtle leakage
  inconsistency the original version had (the old static column was frozen from the Step 8
  split, not recomputed against the actual CV fold boundaries used to evaluate the model).
- **Step 14**: out-of-fold CV metrics vs. both baselines.
- **Step 15**: feature importance on a final model refit on all data -- checks the model isn't
  leaning on memorized coordinates or the noisy `historical_avg_speed_kmh` feature flagged in Step 9.

In [16]:
import xgboost as xgb
from sklearn.model_selection import GroupKFold
import joblib

full_df = pd.read_csv(os.path.join(OUT_DIR, "full_dataset.csv"))
print(f"Loaded full_dataset.csv: {len(full_df)} rows, {full_df.trip_id.nunique()} distinct trips")

# ---- Step 11: model feature list ----
CATEGORICAL_COLS = ["src_zone_id", "dst_zone_id", "od_zone_pair_id", "day_of_week", "traffic_period_bucket"]
NUMERIC_COLS = [
    "haversine_distance_km", "bearing_degrees",
    "osrm_route_distance_km", "osrm_free_flow_duration_sec", "route_directness_ratio",
    "hour_sin", "hour_cos",
    "is_friday", "is_saturday", "is_weekend", "is_holiday", "rush_hour_flag",
]
MODEL_FEATURE_COLS = CATEGORICAL_COLS + NUMERIC_COLS + ["historical_avg_speed_kmh"]

y_log = full_df["log_duration"].to_numpy()
y_true_sec = full_df["duration_seconds"].to_numpy()
groups = full_df["trip_id"].to_numpy()
print(f"Model features ({len(MODEL_FEATURE_COLS)}): {MODEL_FEATURE_COLS}")

# ---- Step 12: baselines ----
def metrics(y_true, y_pred, name):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"  {name:>28}: MAE={mae:8.1f}s  RMSE={rmse:8.1f}s  MAPE={mape:6.1f}%")

osrm_free = full_df["osrm_free_flow_duration_sec"].to_numpy()
median_slowdown = np.median(y_true_sec / osrm_free)
print(f"\nGlobal median slowdown factor: {median_slowdown:.3f}")
print("Baselines:")
metrics(y_true_sec, osrm_free, "naive (OSRM free-flow)")
metrics(y_true_sec, osrm_free * median_slowdown, "calibrated (x median slowdown)")

# ---- historical-feature lookup helpers (per-fold, leakage-safe -- shared with Step 16) ----
MIN_SUPPORT = 3  # minimum DISTINCT TRIPS behind a historical-speed cell before it is trusted

def compute_historical_lookup(train_df):
    d = train_df.copy()
    d["true_speed_kmh"] = d["osrm_route_distance_km"] / (d["duration_seconds"] / 3600.0)
    agg = dict(speed=("true_speed_kmh", "median"), n_trips=("trip_id", "nunique"))
    lvl1 = d.groupby(["od_zone_pair_id", "traffic_period_bucket"]).agg(**agg)
    lvl2 = d.groupby(["od_zone_pair_id"]).agg(**agg)
    lvl3 = d.groupby(["traffic_period_bucket"]).agg(**agg)
    global_speed = d["true_speed_kmh"].median()
    return lvl1, lvl2, lvl3, global_speed

def period_factor(bucket, lvl3, global_speed):
    """How much faster/slower this time of day is than the all-day median (1.0 if unsupported)."""
    if bucket in lvl3.index and lvl3.loc[bucket, "n_trips"] >= MIN_SUPPORT:
        return lvl3.loc[bucket, "speed"] / global_speed
    return 1.0

def lookup_historical(row, lvl1, lvl2, lvl3, global_speed):
    key1 = (row["od_zone_pair_id"], row["traffic_period_bucket"])
    if key1 in lvl1.index and lvl1.loc[key1, "n_trips"] >= MIN_SUPPORT:
        return lvl1.loc[key1, "speed"], "od_pair+period"
    key2 = row["od_zone_pair_id"]
    if key2 in lvl2.index and lvl2.loc[key2, "n_trips"] >= MIN_SUPPORT:
        return lvl2.loc[key2, "speed"] * period_factor(row["traffic_period_bucket"], lvl3, global_speed), "od_pair_only"
    key3 = row["traffic_period_bucket"]
    if key3 in lvl3.index and lvl3.loc[key3, "n_trips"] >= MIN_SUPPORT:
        return lvl3.loc[key3, "speed"], "period_only"
    return global_speed, "global_fallback"

def build_fold_matrices(train_d, val_d):
    """Recompute historical_avg_speed_kmh from train_d only, then one-hot-safe category
    alignment for both -- returns (X_train, X_val, val_level_series)."""
    lvl1, lvl2, lvl3, global_speed = compute_historical_lookup(train_d)
    for d in (train_d, val_d):
        res = d.apply(lambda r: lookup_historical(r, lvl1, lvl2, lvl3, global_speed), axis=1)
        d["historical_avg_speed_kmh"] = res.apply(lambda x: x[0])
        d["historical_feature_level"] = res.apply(lambda x: x[1])

    X_train = train_d[MODEL_FEATURE_COLS].copy()
    X_val = val_d[MODEL_FEATURE_COLS].copy()
    for c in CATEGORICAL_COLS:
        all_cats = pd.concat([X_train[c], X_val[c]]).astype(str).unique()
        X_train[c] = pd.Categorical(X_train[c].astype(str), categories=all_cats)
        X_val[c] = pd.Categorical(X_val[c].astype(str), categories=all_cats)
    for c in ["is_friday", "is_saturday", "is_weekend", "is_holiday", "rush_hour_flag"]:
        X_train[c] = X_train[c].astype(int)
        X_val[c] = X_val[c].astype(int)
    return X_train, X_val, val_d["historical_feature_level"]

# ---- Step 13: XGBoost, grouped 5-fold CV, historical feature recomputed per fold ----
N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)
fold_splits = list(gkf.split(full_df, groups=groups))
oof_pred_sec = np.zeros(len(full_df))
fallback_level_counts = {}

for fold_num, (train_idx, val_idx) in enumerate(fold_splits, start=1):
    train_d = full_df.iloc[train_idx].copy()
    val_d = full_df.iloc[val_idx].copy()
    X_train, X_val, val_level = build_fold_matrices(train_d, val_d)
    for lvl_name, cnt in val_level.value_counts().items():
        fallback_level_counts[lvl_name] = fallback_level_counts.get(lvl_name, 0) + cnt

    model = xgb.XGBRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        tree_method="hist", enable_categorical=True, random_state=42,
    )
    model.fit(X_train, y_log[train_idx])
    pred_log = model.predict(X_val)
    oof_pred_sec[val_idx] = np.exp(pred_log)
    fold_mae = np.mean(np.abs(y_true_sec[val_idx] - np.exp(pred_log)))
    print(f"Fold {fold_num}/{N_FOLDS}: {len(val_idx)} val rows, "
          f"{len(set(groups[val_idx]))} distinct trips, fold MAE={fold_mae:.1f}s")

print(f"\nhistorical_feature_level breakdown across all out-of-fold predictions: {fallback_level_counts}")

# ---- Step 14: evaluate CV performance vs baselines ----
print("\nModel (grouped 5-fold CV, out-of-fold, historical feature recomputed per fold) vs baselines:")
metrics(y_true_sec, osrm_free, "naive (OSRM free-flow)")
metrics(y_true_sec, osrm_free * median_slowdown, "calibrated (x median slowdown)")
metrics(y_true_sec, oof_pred_sec, "XGBoost (out-of-fold)")

# ---- final model fit on ALL data (historical feature also from ALL data) + Step 15 feature importance ----
lvl1_all, lvl2_all, lvl3_all, global_speed_all = compute_historical_lookup(full_df)
res_all = full_df.apply(lambda r: lookup_historical(r, lvl1_all, lvl2_all, lvl3_all, global_speed_all), axis=1)
full_df["historical_avg_speed_kmh"] = res_all.apply(lambda x: x[0])

X_all = full_df[MODEL_FEATURE_COLS].copy()
for c in CATEGORICAL_COLS:
    X_all[c] = X_all[c].astype(str).astype("category")
for c in ["is_friday", "is_saturday", "is_weekend", "is_holiday", "rush_hour_flag"]:
    X_all[c] = X_all[c].astype(int)

final_model = xgb.XGBRegressor(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", enable_categorical=True, random_state=42,
)
final_model.fit(X_all, y_log)

importances = pd.Series(final_model.feature_importances_, index=MODEL_FEATURE_COLS).sort_values(ascending=False)
print("\nFeature importance (final model, trained on all data):")
print(importances.to_string())

model_path = os.path.join(MODEL_DIR, "xgb_model.joblib")
joblib.dump({"model": final_model, "feature_cols": MODEL_FEATURE_COLS, "categorical_cols": CATEGORICAL_COLS}, model_path)
print(f"\nSaved final model -> {model_path}")

Loaded full_dataset.csv: 7926 rows, 255 distinct trips
Model features (18): ['src_zone_id', 'dst_zone_id', 'od_zone_pair_id', 'day_of_week', 'traffic_period_bucket', 'haversine_distance_km', 'bearing_degrees', 'osrm_route_distance_km', 'osrm_free_flow_duration_sec', 'route_directness_ratio', 'hour_sin', 'hour_cos', 'is_friday', 'is_saturday', 'is_weekend', 'is_holiday', 'rush_hour_flag', 'historical_avg_speed_kmh']

Global median slowdown factor: 2.649
Baselines:
        naive (OSRM free-flow): MAE=   708.4s  RMSE=  1451.7s  MAPE=  60.0%
  calibrated (x median slowdown): MAE=   505.4s  RMSE=  1122.9s  MAPE=  64.6%
Fold 1/5: 1585 val rows, 50 distinct trips, fold MAE=578.0s
Fold 2/5: 1587 val rows, 51 distinct trips, fold MAE=418.0s
Fold 3/5: 1585 val rows, 52 distinct trips, fold MAE=572.1s
Fold 4/5: 1585 val rows, 51 distinct trips, fold MAE=569.1s
Fold 5/5: 1584 val rows, 51 distinct trips, fold MAE=492.4s

historical_feature_level breakdown across all out-of-fold predictions: {'od_p

## Step 16 -- Algorithm comparison: is XGBoost actually the best choice?

Up to now XGBoost was chosen by *reasoning* (GBDTs suit small, mixed categorical/numeric
tabular data better than a neural net) but never empirically tested against alternatives.
This runs Ridge Linear Regression, Random Forest, and XGBoost under the identical grouped
5-fold CV split -- and, to keep the comparison fair, Ridge and Random Forest now also use the
same per-fold-recomputed `historical_avg_speed_kmh` feature as Step 13's XGBoost, rather than
a static column.

**LightGBM was attempted and dropped**: confirmed via isolated testing that LightGBM 4.7.0
crashes with a native access violation in this Windows/Python 3.12 environment as soon as
`pandas` is imported in the process -- reproducible even with plain numpy arrays and no
DataFrame involved at all. This is an environment DLL conflict, not a data or code issue.

In [17]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

def metrics_tuple(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return mae, rmse, mape

def build_fold_onehot(train_d, val_d):
    """Same per-fold historical recomputation as build_fold_matrices, but one-hot encoded
    (for Ridge/Random Forest, which don't take pandas 'category' dtype natively)."""
    lvl1, lvl2, lvl3, global_speed = compute_historical_lookup(train_d)
    for d in (train_d, val_d):
        res = d.apply(lambda r: lookup_historical(r, lvl1, lvl2, lvl3, global_speed), axis=1)
        d["historical_avg_speed_kmh"] = res.apply(lambda x: x[0])

    Xtr = pd.get_dummies(train_d[MODEL_FEATURE_COLS], columns=CATEGORICAL_COLS)
    Xval = pd.get_dummies(val_d[MODEL_FEATURE_COLS], columns=CATEGORICAL_COLS)
    Xval = Xval.reindex(columns=Xtr.columns, fill_value=0)
    for c in ["is_friday", "is_saturday", "is_weekend", "is_holiday", "rush_hour_flag"]:
        Xtr[c] = Xtr[c].astype(int)
        Xval[c] = Xval[c].astype(int)
    return Xtr, Xval

comparison = {}

# Ridge Linear Regression (one-hot, scaled, historical feature recomputed per fold)
oof = np.zeros(len(full_df))
for train_idx, val_idx in fold_splits:
    train_d = full_df.iloc[train_idx].copy()
    val_d = full_df.iloc[val_idx].copy()
    Xtr_raw, Xval_raw = build_fold_onehot(train_d, val_d)
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr_raw)
    Xval = scaler.transform(Xval_raw)
    m = Ridge(alpha=1.0, random_state=42)
    m.fit(Xtr, y_log[train_idx])
    oof[val_idx] = np.exp(m.predict(Xval))
comparison["Ridge Linear Regression"] = metrics_tuple(y_true_sec, oof)

# Random Forest (one-hot, historical feature recomputed per fold)
oof = np.zeros(len(full_df))
for train_idx, val_idx in fold_splits:
    train_d = full_df.iloc[train_idx].copy()
    val_d = full_df.iloc[val_idx].copy()
    Xtr_raw, Xval_raw = build_fold_onehot(train_d, val_d)
    m = RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=3, random_state=42, n_jobs=-1)
    m.fit(Xtr_raw, y_log[train_idx])
    oof[val_idx] = np.exp(m.predict(Xval_raw))
comparison["Random Forest"] = metrics_tuple(y_true_sec, oof)

# XGBoost (already computed in Step 13 as oof_pred_sec, same fold splits/methodology) + baselines
comparison["XGBoost"] = metrics_tuple(y_true_sec, oof_pred_sec)
comparison["Baseline: naive OSRM"] = metrics_tuple(y_true_sec, osrm_free)
comparison["Baseline: calibrated"] = metrics_tuple(y_true_sec, osrm_free * median_slowdown)

print(f"\n{'Model':<28} {'MAE(s)':>10} {'RMSE(s)':>10} {'MAPE(%)':>10}")
print("=" * 62)
for name, (mae, rmse, mape) in sorted(comparison.items(), key=lambda x: x[1][0]):
    print(f"{name:<28} {mae:>10.1f} {rmse:>10.1f} {mape:>10.1f}")

n_od_pairs = full_df["od_zone_pair_id"].nunique()
print(f"\nNote: with the fixed-grid zoning, od_zone_pair_id has {n_od_pairs} distinct levels; "
      "one-hot encoding this many sparse zone-pair columns for a modest number of real trips "
      "is exactly the kind of high-dimensional-sparse setup that can destabilize plain linear "
      "regression once the target is exponentiated back out of log-space -- watch Ridge's "
      "numbers below for that.")


Model                            MAE(s)    RMSE(s)    MAPE(%)
Random Forest                     465.4     1047.8       55.6
Baseline: calibrated              505.4     1122.9       64.6
XGBoost                           525.9     1234.5       61.0
Baseline: naive OSRM              708.4     1451.7       60.0
Ridge Linear Regression          2330.0    57911.0      104.2

Note: with the fixed-grid zoning, od_zone_pair_id has 79 distinct levels; one-hot encoding this many sparse zone-pair columns for a modest number of real trips is exactly the kind of high-dimensional-sparse setup that can destabilize plain linear regression once the target is exponentiated back out of log-space -- watch Ridge's numbers below for that.


## Step 17 -- Persist everything inference needs (not just the model)

This is the key point for the question *"how does inference work for a brand-new src/dst,
since raw lat/lon isn't a training feature?"* -- raw lat/lon was never fed to the trees as a
split candidate, but it is absolutely still used at inference time, as the **input** to the
same feature-generation functions used during training:

- `haversine_distance_km` / `bearing_degrees`: plain formulas, computable for *any* two
  coordinates, no dependency on training data at all.
- `osrm_route_distance_km` / `osrm_free_flow_duration_sec`: a live OSRM query on the raw new
  coordinates -- OSRM works for any point inside the Bangladesh map extract, not just points
  that happened to appear in training.
- `src_zone_id` / `dst_zone_id`: **the fixed grid parameters from Step 6** (`min_lat`,
  `min_lon`, `lat_step`, `lon_step`, `n_rows`, `n_cols`) applied to the raw new coordinates.
  Unlike the old KMeans version, there's no model object to keep in sync with the training
  rows -- just 6 numbers, which is exactly the stability benefit the grid switch was for.
- time features: computed directly from the query's timestamp, same formulas, no dependency
  on training data.
- `historical_avg_speed_kmh`: a lookup in the **saved aggregation tables** (with the same
  fallback hierarchy from Step 9), keyed on the *new* row's `od_zone_pair_id` +
  `traffic_period_bucket`.

So the deployable unit isn't just `xgb_model.joblib` -- it's the model(s) **plus** the grid
parameters **plus** the historical lookup tables **plus** the exact feature schema. All of it
is (re)computed on the **entire** `full_dataset.csv` here (all 89 traces / current trip
count), since there's no more held-out set to protect once this is the deployed artifact --
Step 8/9's train-only split was for honest *evaluation*, this is for *production*.

In [18]:
# Historical lookup tables on ALL data were already computed above (lvl1_all, lvl2_all,
# lvl3_all, global_speed_all) as part of Step 15's final-model fit -- reuse them directly.

# XGBoost was already fit on ALL data above (final_model) -- reuse it as the deployed XGBoost.
xgb_deploy = final_model

# Random Forest wasn't fit on all data yet -- do that now (one-hot, same as Step 16).
X_onehot_all = pd.get_dummies(full_df[MODEL_FEATURE_COLS], columns=CATEGORICAL_COLS)
for c in ["is_friday", "is_saturday", "is_weekend", "is_holiday", "rush_hour_flag"]:
    X_onehot_all[c] = X_onehot_all[c].astype(int)

rf_deploy = RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_deploy.fit(X_onehot_all, y_log)

CAT_CATEGORIES = {c: X_all[c].cat.categories for c in CATEGORICAL_COLS}
ONEHOT_COLUMNS = list(X_onehot_all.columns)

bundle = {
    "rf_model": rf_deploy, "xgb_model": xgb_deploy,
    "zone_method": "fixed_grid",
    "grid_params": {
        "min_lat": GRID_MIN_LAT, "min_lon": GRID_MIN_LON,
        "lat_step": GRID_LAT_STEP, "lon_step": GRID_LON_STEP,
        "n_rows": N_ROWS, "n_cols": N_COLS,
    },
    "lvl1": lvl1_all, "lvl2": lvl2_all, "lvl3": lvl3_all, "global_speed": global_speed_all,
    "min_support": MIN_SUPPORT,
    "feature_cols": MODEL_FEATURE_COLS, "categorical_cols": CATEGORICAL_COLS,
    "cat_categories": CAT_CATEGORIES, "onehot_columns": ONEHOT_COLUMNS,
    "osrm_base": OSRM_BASE,
}
bundle_path = os.path.join(MODEL_DIR, "inference_bundle.joblib")
joblib.dump(bundle, bundle_path)
print(f"Saved inference bundle (models + fixed-grid params + historical tables + schema) -> {bundle_path}")

Saved inference bundle (models + fixed-grid params + historical tables + schema) -> d:\Academic\4-1\Capstone\GPS_TRACE_ML\trained-model\inference_bundle.joblib


## Step 18 -- The inference-time feature pipeline, demonstrated live

`build_feature_row()` below takes only `(src_lat, src_lon, dst_lat, dst_lon, query_time)` --
exactly the 3 raw inputs a real user would supply -- and reconstructs the full feature row
using the artifacts from Step 17. It works for coordinates that never appeared anywhere in
training, because every step (formulas, live OSRM query, the fixed-grid arithmetic, lookup
table joins) is a *function of the raw input*, not a lookup into training rows.

Two demos below:
1. A known real trip from the data (Zigatola -> Dhanmondi), to sanity-check against the real
   491s duration.
2. A genuinely novel src/dst pair whose `od_zone_pair_id` was never seen in training at all --
   this is what actually happens when the historical-feature fallback kicks in, and it also
   surfaces a real difference between the two models: one-hot encoding (Random Forest) silently
   zeroes out the unseen zone-pair's dummy column, while XGBoost's native categorical dtype
   represents it as an explicit missing value that its tree splits handle directly. Worth
   watching for a large disagreement between the two models specifically on out-of-distribution
   queries like this.

In [19]:
FIXED_HOLIDAYS_MD_INF = FIXED_HOLIDAYS_MD  # reuse from Step 7

def grid_cell_from_bundle(lat, lon, b):
    """Same arithmetic as Step 6's grid_cell(), but reads the 6 stored parameters from the
    bundle instead of the notebook's live globals -- this is what makes it portable to a
    fresh process (e.g. a different notebook) that only has the .joblib file."""
    gp = b["grid_params"]
    r = min(int((lat - gp["min_lat"]) / gp["lat_step"]), gp["n_rows"] - 1)
    c = min(int((lon - gp["min_lon"]) / gp["lon_step"]), gp["n_cols"] - 1)
    return r * gp["n_cols"] + c

def build_feature_row(src_lat, src_lon, dst_lat, dst_lon, query_time, b=bundle):
    """Turn a raw (src, dst, time) query into the exact feature row the models expect.
    Raw lat/lon are used HERE -- as inputs to formulas and to the fixed grid -- they are just
    never fed to the trees directly as split candidates."""
    dt = pd.to_datetime(query_time)

    haversine_distance_km = haversine_km(src_lat, src_lon, dst_lat, dst_lon)
    bearing_degrees = bearing_deg(src_lat, src_lon, dst_lat, dst_lon)

    url = f"{b['osrm_base']}/route/v1/driving/{src_lon},{src_lat};{dst_lon},{dst_lat}?overview=false"
    route = requests.get(url, timeout=5).json()["routes"][0]
    osrm_route_distance_km = route["distance"] / 1000.0
    osrm_free_flow_duration_sec = route["duration"]
    route_directness_ratio = (haversine_distance_km / osrm_route_distance_km) if osrm_route_distance_km > 0 else np.nan

    src_zone_id = grid_cell_from_bundle(src_lat, src_lon, b)
    dst_zone_id = grid_cell_from_bundle(dst_lat, dst_lon, b)
    od_zone_pair_id = f"{src_zone_id}_{dst_zone_id}"

    hour = dt.hour + dt.minute / 60.0 + dt.second / 3600.0
    day_of_week = dt.day_name()
    is_friday, is_saturday = day_of_week == "Friday", day_of_week == "Saturday"
    is_weekend = is_friday or is_saturday
    rush_hour_flag = (8 <= hour < 10) or (17 <= hour < 21)
    bucket = traffic_bucket(hour)
    is_holiday = (dt.month, dt.day) in FIXED_HOLIDAYS_MD_INF

    lvl1, lvl2, lvl3 = b["lvl1"], b["lvl2"], b["lvl3"]
    key1 = (od_zone_pair_id, bucket)
    od_pair_seen = od_zone_pair_id in lvl2.index
    if key1 in lvl1.index and lvl1.loc[key1, "n_trips"] >= b["min_support"]:
        historical_avg_speed_kmh, level = lvl1.loc[key1, "speed"], "od_pair+period"
    elif od_pair_seen and lvl2.loc[od_zone_pair_id, "n_trips"] >= b["min_support"]:
        supported = bucket in lvl3.index and lvl3.loc[bucket, "n_trips"] >= b["min_support"]
        factor = lvl3.loc[bucket, "speed"] / b["global_speed"] if supported else 1.0
        historical_avg_speed_kmh, level = lvl2.loc[od_zone_pair_id, "speed"] * factor, "od_pair_only"
    elif bucket in lvl3.index and lvl3.loc[bucket, "n_trips"] >= b["min_support"]:
        historical_avg_speed_kmh, level = lvl3.loc[bucket, "speed"], "period_only"
    else:
        historical_avg_speed_kmh, level = b["global_speed"], "global_fallback"

    row = {
        "src_zone_id": src_zone_id, "dst_zone_id": dst_zone_id, "od_zone_pair_id": od_zone_pair_id,
        "day_of_week": day_of_week, "traffic_period_bucket": bucket,
        "haversine_distance_km": haversine_distance_km, "bearing_degrees": bearing_degrees,
        "osrm_route_distance_km": osrm_route_distance_km,
        "osrm_free_flow_duration_sec": osrm_free_flow_duration_sec,
        "route_directness_ratio": route_directness_ratio,
        "hour_sin": np.sin(2 * np.pi * hour / 24.0), "hour_cos": np.cos(2 * np.pi * hour / 24.0),
        "is_friday": int(is_friday), "is_saturday": int(is_saturday), "is_weekend": int(is_weekend),
        "is_holiday": int(is_holiday), "rush_hour_flag": int(rush_hour_flag),
        "historical_avg_speed_kmh": historical_avg_speed_kmh,
    }
    return row, level, od_pair_seen

def predict_travel_time(src_lat, src_lon, dst_lat, dst_lon, query_time, b=bundle):
    """Returns predicted duration in MINUTES for both models."""
    row, level, od_pair_seen = build_feature_row(src_lat, src_lon, dst_lat, dst_lon, query_time, b)

    xrow = pd.DataFrame([row])
    for c in b["categorical_cols"]:
        xrow[c] = pd.Categorical([str(row[c])], categories=b["cat_categories"][c])
    for c in ["is_friday", "is_saturday", "is_weekend", "is_holiday", "rush_hour_flag"]:
        xrow[c] = xrow[c].astype(int)
    xrow = xrow[b["feature_cols"]]
    xgb_pred_sec = float(np.exp(b["xgb_model"].predict(xrow)[0]))

    orow = pd.get_dummies(pd.DataFrame([row]), columns=b["categorical_cols"])
    orow = orow.reindex(columns=b["onehot_columns"], fill_value=0)
    rf_pred_sec = float(np.exp(b["rf_model"].predict(orow)[0]))

    return {"xgboost": xgb_pred_sec / 60.0, "random_forest": rf_pred_sec / 60.0}, level, od_pair_seen, row


print("=== Demo 1: known real route (Zigatola -> Dhanmondi) ===")
preds, level, seen, row = predict_travel_time(23.7388773, 90.3756685, 23.7564776, 90.3622345, "2024-12-09T01:20:05")
print(f"od_zone_pair_id={row['od_zone_pair_id']}  historical_feature_level={level}  zone-pair seen in training={seen}")
print(f"Predicted duration (minutes): {preds}")
print("Actual real duration for this exact trip: 491s (~8.2 min)")

print("\n=== Demo 2: genuinely novel src/dst pair ===")
preds, level, seen, row = predict_travel_time(23.725268988907633, 90.39161055488904, 23.781477356287553, 90.3517412364306, "2026-09-10T12:55:00")
print(f"od_zone_pair_id={row['od_zone_pair_id']}  historical_feature_level={level}  zone-pair seen in training={seen}")
print(f"Predicted duration (minutes): {preds}")
print("Note the sizable disagreement between the two models on this out-of-distribution query --"
      " exactly the risk of extrapolating beyond the real trips this was trained on.")

=== Demo 1: known real route (Zigatola -> Dhanmondi) ===
od_zone_pair_id=7_6  historical_feature_level=od_pair_only  zone-pair seen in training=True
Predicted duration (minutes): {'xgboost': 7.088421630859375, 'random_forest': 11.677603356790032}
Actual real duration for this exact trip: 491s (~8.2 min)

=== Demo 2: genuinely novel src/dst pair ===
od_zone_pair_id=7_6  historical_feature_level=od_pair+period  zone-pair seen in training=True
Predicted duration (minutes): {'xgboost': 47.6183837890625, 'random_forest': 28.73828830186378}
Note the sizable disagreement between the two models on this out-of-distribution query -- exactly the risk of extrapolating beyond the real trips this was trained on.


## Step 19 -- Try your own src/dst/time (predicted time in minutes)

Edit the 5 values below and re-run this cell -- no need to touch anything else. Reuses the
exact same `predict_travel_time()` pipeline from Step 18 (live OSRM query + fitted zone model
+ historical lookup), so it works for any coordinates, not just ones seen in training.

In [20]:
# ---- Edit these 5 values, then re-run this cell ----
CUSTOM_SRC_LAT = 23.90672
CUSTOM_SRC_LON = 90.41704
CUSTOM_DST_LAT = 23.77431
CUSTOM_DST_LON = 90.41207
CUSTOM_QUERY_TIME = "2026-09-23T06:00:00"  # ISO format, e.g. "YYYY-MM-DDTHH:MM:SS"
# -----------------------------------------------------

preds, level, seen, row = predict_travel_time(
    CUSTOM_SRC_LAT, CUSTOM_SRC_LON, CUSTOM_DST_LAT, CUSTOM_DST_LON, CUSTOM_QUERY_TIME
)

print(f"Query: ({CUSTOM_SRC_LAT}, {CUSTOM_SRC_LON}) -> ({CUSTOM_DST_LAT}, {CUSTOM_DST_LON})  at {CUSTOM_QUERY_TIME}")
print(f"  src_zone_id={row['src_zone_id']}  dst_zone_id={row['dst_zone_id']}  od_zone_pair_id={row['od_zone_pair_id']}")
print(f"  day_of_week={row['day_of_week']}  traffic_period_bucket={row['traffic_period_bucket']}  rush_hour={bool(row['rush_hour_flag'])}")
print(f"  haversine_distance_km={row['haversine_distance_km']:.2f}  osrm_route_distance_km={row['osrm_route_distance_km']:.2f}")
print(f"  historical_feature_level={level}  (od_zone_pair seen in training={seen})")
print()
print("Predicted travel time:")
print(f"  XGBoost      : {preds['xgboost']:.1f} minutes")
print(f"  Random Forest: {preds['random_forest']:.1f} minutes")
if not seen:
    print()
    print("Note: this exact od_zone_pair_id was never seen in training -- treat this prediction "
          "with more caution, and watch for the two models disagreeing more than usual.")

Query: (23.90672, 90.41704) -> (23.77431, 90.41207)  at 2026-09-23T06:00:00
  src_zone_id=12  dst_zone_id=7  od_zone_pair_id=12_7
  day_of_week=Wednesday  traffic_period_bucket=morning_offpeak  rush_hour=False
  haversine_distance_km=14.73  osrm_route_distance_km=19.23
  historical_feature_level=od_pair_only  (od_zone_pair seen in training=True)

Predicted travel time:
  XGBoost      : 58.1 minutes
  Random Forest: 44.9 minutes


## Step 20 -- Predict every leg in route_legs.xlsx (XGBoost + Random Forest, minutes)

Loads `route_legs.xlsx`, groups rows into distinct routes (same `Route (Driver | Vehicle)` +
`Arrival Time`, per the Notes sheet's own rule), and predicts each leg's travel time with both
models -- chaining the departure clock forward leg by leg within each route.

**Assumption made here** (not specified this time, unlike the earlier "10 PM" run): each
route's *own* `Arrival Time` column value (e.g. "10:00 PM", "12:00 AM") is used as that
route's actual departure time for leg 1, rather than assuming one fixed time for every route.
This is more faithful to the sheet's own data than picking a single time for everything --
adjust `REFERENCE_DATE` below if a specific calendar date matters.

Adds 6 columns: `predicted_departure_time`, `xgboost_minutes`, `random_forest_minutes`,
`od_zone_pair_seen_in_training`, `historical_feature_level`, `prediction_error` (filled only
for rows with missing coordinates in the source sheet). Saves to
`route_legs_with_predictions.xlsx` (original file untouched), keeping the `Notes` sheet.

In [21]:
ROUTE_LEGS_PATH = os.path.join(PROJECT_DIR, "route_legs.xlsx")
ROUTE_LEGS_OUT_PATH = os.path.join(PROJECT_DIR, "route_legs_with_predictions_23.xlsx")
REFERENCE_DATE = "2026-09-10"  # calendar date used for every route\'s Arrival Time

xls = pd.ExcelFile(ROUTE_LEGS_PATH)
legs_df = pd.read_excel(xls, sheet_name="Route Legs")
notes_df = pd.read_excel(xls, sheet_name="Notes")
print(f"Loaded {len(legs_df)} legs from route_legs.xlsx")

legs_df["_route_key"] = legs_df["Route (Driver | Vehicle)"].astype(str) + " || " + legs_df["Arrival Time"].astype(str)
legs_df = legs_df.sort_values(["_route_key", "Leg #"]).reset_index(drop=False)
print(f"Grouped into {legs_df["_route_key"].nunique()} distinct routes (by driver/vehicle + Arrival Time)")

coord_cols = ["source_lat", "source_lang", "destination_lat", "destination_lang"]
results = {
    "predicted_departure_time": [None] * len(legs_df),
    "xgboost_minutes": [None] * len(legs_df),
    "random_forest_minutes": [None] * len(legs_df),
    "od_zone_pair_seen_in_training": [None] * len(legs_df),
    "historical_feature_level": [None] * len(legs_df),
    "prediction_error": [None] * len(legs_df),
}

n_errors = 0
for route_key, group in legs_df.groupby("_route_key", sort=False):
    arrival_time_str = group["Arrival Time"].iloc[0]
    try:
        time_of_day = pd.to_datetime(arrival_time_str, format="%I:%M %p").time()
        current_time = pd.Timestamp.combine(pd.to_datetime(REFERENCE_DATE).date(), time_of_day)
    except Exception:
        current_time = pd.to_datetime(REFERENCE_DATE + "T23:00:00")  # fallback if unparsable

    for pos in group.index:
        r = legs_df.loc[pos]
        results["predicted_departure_time"][pos] = current_time

        if r[coord_cols].isna().any():
            results["prediction_error"][pos] = "missing source/destination coordinates in spreadsheet"
            n_errors += 1
            continue

        try:
            preds, level, seen, feat_row = predict_travel_time(
                r["source_lat"], r["source_lang"], r["destination_lat"], r["destination_lang"], current_time
            )
        except Exception as e:
            results["prediction_error"][pos] = f"{type(e).__name__}: {e}"
            n_errors += 1
            continue

        results["xgboost_minutes"][pos] = round(preds["xgboost"], 1)
        results["random_forest_minutes"][pos] = round(preds["random_forest"], 1)
        results["od_zone_pair_seen_in_training"][pos] = seen
        results["historical_feature_level"][pos] = level
        current_time = current_time + pd.Timedelta(minutes=preds["xgboost"])

for col, vals in results.items():
    legs_df[col] = vals

legs_df = legs_df.sort_values("index").drop(columns=["index", "_route_key"]).reset_index(drop=True)

print(f"\n{n_errors} leg(s) could not be predicted (see prediction_error column)")
ok = legs_df["xgboost_minutes"].notna()
print(f"{ok.sum()}/{len(legs_df)} legs predicted successfully")
n_unseen = (legs_df.loc[ok, "od_zone_pair_seen_in_training"] == False).sum()
print(f"{n_unseen}/{ok.sum()} successful legs have an OD zone-pair never seen in training")
print(f"Total predicted time -- XGBoost: {legs_df.loc[ok, "xgboost_minutes"].sum():.1f} min, "
      f"Random Forest: {legs_df.loc[ok, "random_forest_minutes"].sum():.1f} min")

print("\nSample:")
print(legs_df[["source", "destination", "Leg #", "predicted_departure_time",
               "xgboost_minutes", "random_forest_minutes",
               "od_zone_pair_seen_in_training", "historical_feature_level"]].head(10).to_string())

with pd.ExcelWriter(ROUTE_LEGS_OUT_PATH, engine="openpyxl") as writer:
    legs_df.to_excel(writer, sheet_name="Route Legs", index=False)
    notes_df.to_excel(writer, sheet_name="Notes", index=False)
print(f"\nSaved -> {ROUTE_LEGS_OUT_PATH}")

Loaded 135 legs from route_legs.xlsx
Grouped into 49 distinct routes (by driver/vehicle + Arrival Time)

4 leg(s) could not be predicted (see prediction_error column)
131/135 legs predicted successfully
1/131 successful legs have an OD zone-pair never seen in training
Total predicted time -- XGBoost: 1813.6 min, Random Forest: 1803.2 min

Sample:
                                                           source                                                     destination  Leg #      predicted_departure_time  xgboost_minutes  random_forest_minutes od_zone_pair_seen_in_training historical_feature_level
0                                             Mazar Road Bus Stop                                                       Technical      1 2026-09-10 22:00:00.000000000              5.8                    5.3                          True           od_pair+period
1                                                       Technical                                                 Darussalam ro